In [184]:
import sys, os
import pandas as pd
from utils.misc import cols_to_front
from pathlib import Path
from datetime import datetime
import shutil
import nbformat
from nbconvert.preprocessors import ExecutePreprocessor
import subprocess
import yaml
with open('config/config.yaml', 'r') as file:
    config = yaml.safe_load(file)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

(Path("runs") / ".gitkeep").touch(exist_ok=True)

In [185]:
run_name = (
    f"{config.get('exp_name','NA').upper()}"
    f"_rd-{str(config.get('reduce_dim','NA')).lower()}"
    f"_meth-{config.get('reduce_dim_method','NA')}"
    f"_dim-{config.get('n_dim','NA')}"
    f"_idf-{str(config.get('idf','NA')).lower()}"
)

# Add timestamp for uniqueness
run_name = f"{run_name}_{timestamp}"

# Run directory
RUN_DIR = Path(config["run_dir"]) / run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(RUN_DIR)

runs\GPT52-RAG_rd-true_meth-svd_dim-50_idf-false_2026-03-05_14-37


In [186]:
#save current git commit in yaml
def git(cmd):
    return subprocess.check_output(cmd, shell=True).decode().strip()

# add to config
config["git_branch"] = git("git rev-parse --abbrev-ref HEAD")
config["git_commit"] =  git("git rev-parse --short HEAD")

with open(RUN_DIR / "config.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

# Run the pipeline

In [187]:
# Run all the pipeline notebooks sequentially
NOTEBOOK_DIR = Path(".")
notebooks = [
    "data_processing.ipynb",
    'vectorize_ai_descriptors.ipynb',
    'ingredients_matching.ipynb',
    'data_exploration.ipynb',
]

# -----------------------------------
# Execute notebooks sequentially
# -----------------------------------
for nb_name in notebooks:
    nb_path = NOTEBOOK_DIR / nb_name
    print(f"\n Executing {nb_name}")

    with open(nb_path) as f:
        nb = nbformat.read(f, as_version=4)

    ep = ExecutePreprocessor(
        timeout=600,          # increase if long runs
        kernel_name="python3"
    )

    ep.preprocess(nb, {"metadata": {"path": NOTEBOOK_DIR}})

    # Overwrite notebook with executed outputs
    with open(nb_path, "w", encoding="utf-8") as f:
        nbformat.write(nb, f)

    print(f"✅ Finished {nb_name}")


 Executing data_processing.ipynb
✅ Finished data_processing.ipynb

 Executing vectorize_ai_descriptors.ipynb
✅ Finished vectorize_ai_descriptors.ipynb

 Executing ingredients_matching.ipynb
✅ Finished ingredients_matching.ipynb

 Executing data_exploration.ipynb
✅ Finished data_exploration.ipynb


# Saving files

In [188]:
data_files = [
    'config/config.yaml',
    'data/merged_ai_descriptors_clean.csv',
    'data/local_df_ai_descriptors.csv',
    'data/merged_ai_descriptors_dummies_filtered.csv',
    'data/merged_ai_descriptors_dummies_full.csv',
    'data/merged_ai_descriptors_clean.csv',
    'data/merged_ai_descriptors.csv',
    'data/flavor_db_descriptors.csv',
    'data/local_df_ai_descriptors.csv',
    'data/all_descriptors.csv'
]

output_files = [
    'output/flavordb_ingredients_top3_matches.xlsx',
    'output/local_ingredients_top3_matches.xlsx',
    'output/cat_clustered_heatmap.png',
    'output/db_jaccard.png',
    'output/cat_tsne.png',
    'output/db_tsne.png',
    "output/category_similarity_top3.xlsx",
    "output/local_flavordb_similarity.xlsx",
    "output/local_flavordb_general_similarity.xlsx"
    
]

In [189]:
# Save data files
data_dir = RUN_DIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# copy files
for file in data_files:
    src = Path(file)
    dst = data_dir / src.name
    shutil.copy2(src, dst)

In [190]:
# Save output files
data_dir = RUN_DIR / "output"
data_dir.mkdir(parents=True, exist_ok=True)

# copy files
for file in output_files:
    src = Path(file)
    dst = data_dir / src.name
    shutil.copy2(src, dst)

# Save overview

In [191]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# -----------------------------
# Input files (your lists)
# -----------------------------
xlsx_files = [
    'output/local_ingredients_top3_matches.xlsx',
    'output/flavordb_ingredients_top3_matches.xlsx'
]

png_files = [
    'output/cat_tsne.png',
    'output/cat_clustered_heatmap.png',
]

cols_to_drop = ["raw_AI_output", "sub_category", "db", "category"]

df1 = pd.read_excel(xlsx_files[0])
df2 = pd.read_excel(xlsx_files[1])

# Drop unwanted columns
df1 = df1.drop(columns=cols_to_drop, errors="ignore")
df2 = df2.drop(columns=cols_to_drop, errors="ignore")

# Selection of rows
ings =['Macrocystis pyrifera', 'Morchella crassipes', 'Picea rubens', 'Allium canadense', 'Rhus glabra', 'Artemisia frigida']
df1 = df1[df1["Nom scientifique"].isin(ings)]   

ings = ['Rice', 'Cedar', 'Sage', 'Apple', "Celery", "Lemon"]
df2 = df2[df2["Nom scientifique"].isin(ings)]   

In [192]:
# Rename matches to common name
merged_df = pd.read_csv('data/local_df_ai_descriptors.csv')

# Create mapping dict
mapping = dict(zip(
    merged_df["Nom scientifique"],
    merged_df["Nom vernaculaire"]
))

# Apply mapping (preserve original if no match)
cols = ["top1_name", "top2_name", "top3_name"]

for col in cols:
    df2[col] = df2[col].map(mapping)

# Combine into one comma-separated column
df2["matches"] = (
    df2[cols]
    .astype(str)
    .apply(lambda row: ", ".join(row.dropna()), axis=1)
)

# Drop some columns to make things more readable
df1 = df1[['Nom scientifique', 'Nom vernaculaire', 'matches', 'descriptor']]
df2 = df2[['Nom scientifique', 'matches', 'descriptor']]


In [193]:
from PIL import Image, ImageDraw, ImageFont
import textwrap

def make_header_image(config, width, dpi=300, pad_x=60, pad_y=30, bg=(255, 255, 255), fg=(0, 0, 0)):
    # Build the header text (use config values; fall back to your requested defaults if missing)
    exp_name = config.get("exp_name", "NA")
    reduce_dim = config.get("reduce_dim", "NA")
    reduce_dim_method = config.get("reduce_dim_method", "NA")
    n_dim = config.get("n_dim", "NA")
    idf = config.get("idf", "NA")

    header_text = (
        f'exp_name: "{exp_name}"    '
        f"reduce_dim: {str(reduce_dim).lower()}    "
        f'reduce_dim_method: "{reduce_dim_method}"    '
        f"n_dim: {n_dim}    "
        f"idf: {str(idf).lower()}"
    )

    # Font: try a nicer TTF, fallback to default
    try:
        # Common on many systems; replace if you have a specific font path
        font = ImageFont.truetype("arial.ttf", size=120)
    except Exception:
        font = ImageFont.load_default()

    # Wrap if needed
    draw_dummy = ImageDraw.Draw(Image.new("RGB", (width, 10), bg))
    max_chars = max(20, int(width / 18))  # heuristic
    lines = textwrap.wrap(header_text, width=max_chars) or [header_text]

    # Measure height
    line_heights = []
    line_widths = []
    for line in lines:
        bbox = draw_dummy.textbbox((0, 0), line, font=font)
        line_widths.append(bbox[2] - bbox[0])
        line_heights.append(bbox[3] - bbox[1])

    text_h = sum(line_heights) + int((len(lines) - 1) * (pad_y * 0.4))
    header_h = pad_y * 2 + text_h

    header = Image.new("RGB", (width, header_h), bg)
    draw = ImageDraw.Draw(header)

    # Optional: subtle separator line at bottom
    draw.line([(0, header_h - 2), (width, header_h - 2)], fill=(220, 220, 220), width=3)

    # Draw lines
    y = pad_y
    for i, line in enumerate(lines):
        bbox = draw.textbbox((0, 0), line, font=font)
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        x = pad_x  # left aligned; change to (width - w)//2 for centered
        draw.text((x, y), line, fill=fg, font=font)
        y += h + int(pad_y * 0.4)

    return header

In [194]:
# -----------------------------
# Load PNG images
# -----------------------------
img1 = Image.open(png_files[0]).convert("RGB")
img2 = Image.open(png_files[1]).convert("RGB")

# -----------------------------
# Convert df.head() → image
# -----------------------------
def df_to_image(df, title="", dpi=300):
    df = df.astype(str)

    nrows, ncols = df.shape

    # -----------------------------
    # Compute column widths from content length
    # -----------------------------
    max_lens = [
        max(
            df[col].str.len().max(),
            len(col)
        )
        for col in df.columns
    ]

    # normalize widths
    total = sum(max_lens)
    col_widths = [l / total for l in max_lens]

    # scale figure size based on total characters
    fig_w = max(16, total * 0.20)
    fig_h = max(6, 1.2 * (nrows + 1))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
    ax.axis("off")

    if title:
        ax.set_title(
            title,
            fontsize=20,
            pad=20,
            fontweight="bold"
        )

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        loc="center",
        cellLoc="left",
        colWidths=col_widths   # <-- KEY FIX
    )

    table.auto_set_font_size(False)
    table.set_fontsize(20)
    table.scale(1, 2.0)   # don't scale width blindly

    # header styling
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_fontsize(22)
            cell.set_text_props(weight='bold')

    tmp = RUN_DIR / "__tmp_df.png"
    fig.savefig(tmp, bbox_inches="tight", dpi=dpi)
    plt.close(fig)

    im = Image.open(tmp).convert("RGB")
    tmp.unlink()

    return im



# Convert to images
df_img1 = df_to_image(df1, title=Path(xlsx_files[0]).name, dpi=config['dpi'])
df_img2 = df_to_image(df2, title=Path(xlsx_files[1]).name, dpi=config['dpi'])

# -----------------------------
# Resize all to same width
# -----------------------------
def resize_to_width(im, w):
    h = int(im.size[1] * w / im.size[0])
    return im.resize((w, h), Image.LANCZOS)

panel_w = min(
    img1.size[0], img2.size[0],
    df_img1.size[0], df_img2.size[0]
)

img1 = resize_to_width(img1, panel_w)
img2 = resize_to_width(img2, panel_w)
df_img1 = resize_to_width(df_img1, panel_w*2)
df_img2 = resize_to_width(df_img2, panel_w*2)

# -----------------------------
# Create composite (2x2 grid)
# -----------------------------
gap = 40
pad = 60
bg = (255, 255, 255)

# Header (config banner)
header = make_header_image(config, width=pad*2 + panel_w*2 + gap, dpi=config["dpi"])

row2_h = max(df_img1.size[1], df_img2.size[1])
row1_h = max(img1.size[1], img2.size[1])

canvas_w = pad*2 + panel_w*2 + gap
canvas_h = (
    header.size[1]              
    + pad*2
    + row1_h
    + gap
    + df_img1.size[1]
    + gap
    + df_img2.size[1]
)


canvas = Image.new("RGB", (canvas_w, canvas_h), bg)

# -----------------------------
# Paste images
# -----------------------------
y0 = header.size[1]

# Header
canvas.paste(header, (0, 0))

# Top row (plots side by side)
canvas.paste(img1, (pad, y0 + pad))
canvas.paste(img2, (pad + panel_w + gap, y0 + pad))

# First dataframe (full width, centered under plots)
y_df1 = y0 + pad + row1_h + gap
canvas.paste(df_img1, (pad, y_df1))

# Second dataframe stacked below
y_df2 = y_df1 + df_img1.size[1] + gap
canvas.paste(df_img2, (pad, y_df2))
# -----------------------------
# Save
# -----------------------------
out_path = RUN_DIR / f"{run_name}_summary.png"

out_path.parent.mkdir(exist_ok=True)
canvas.save(out_path, dpi=(config["dpi"], config["dpi"]))

# Save summary to parent runs folder for quick comparisons
canvas.save(
    Path("runs") / f"{run_name}_summary.png",
    dpi=(config["dpi"], config["dpi"])
)

print(f"Saved composite image to: {out_path}")

Saved composite image to: runs\GPT52-RAG_rd-true_meth-svd_dim-50_idf-false_2026-03-05_14-37\GPT52-RAG_rd-true_meth-svd_dim-50_idf-false_2026-03-05_14-37_summary.png


In [195]:
df= pd.read_excel("output/local_flavordb_similarity.xlsx")
df

,category,flavor_vs_local,flavor_vs_flavor,local_vs_local,flavor_count,local_count
0,plant,0.358974,0.388483,0.456283,45,68
1,fungus,0.606828,0.612392,0.663790,11,14
2,vegetable,0.457377,0.483489,0.539899,41,11
3,root,0.432103,0.505452,0.466371,18,43
4,fruit,0.301432,0.453981,0.413477,81,9
5,herb,0.380897,0.376174,0.838745,51,6
6,additive,0.182427,0.294255,0.586313,25,7
7,flower,0.322625,0.447178,0.566945,9,3


In [196]:
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import textwrap

# -----------------------------
# Load PNG images
# -----------------------------
img1 = Image.open(png_files[0]).convert("RGB")
img2 = Image.open(png_files[1]).convert("RGB")

# -----------------------------
# Convert df → image
# -----------------------------
def df_to_image(df, title="", dpi=300):
    df = df.astype(str)

    nrows, ncols = df.shape

    # Compute column widths from content length
    max_lens = [
        max(df[col].str.len().max(), len(col))
        for col in df.columns
    ]

    total = sum(max_lens) if sum(max_lens) > 0 else 1
    col_widths = [l / total for l in max_lens]

    # scale figure size based on total characters
    fig_w = max(16, total * 0.20)
    fig_h = max(6, 1.05 * (nrows + 1))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
    ax.axis("off")

    if title:
        ax.text(
            0.5, 1.02, title,
            ha="center",
            va="bottom",
            fontsize=20,
            fontweight="bold",
            transform=ax.transAxes
        )

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        loc="center",
        cellLoc="left",
        colWidths=col_widths
    )

    table.auto_set_font_size(False)
    table.set_fontsize(20)
    table.scale(1, 2.0)

    # header styling
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_fontsize(22)
            cell.set_text_props(weight="bold")

    tmp = RUN_DIR / "__tmp_df.png"
    fig.savefig(tmp, bbox_inches="tight", dpi=dpi)
    plt.close(fig)

    im = Image.open(tmp).convert("RGB")
    tmp.unlink(missing_ok=True)

    return im

# -----------------------------
# Header banner
# -----------------------------
def make_header_image(config, width, dpi=300, pad_x=60, pad_y=30, bg=(255, 255, 255), fg=(0, 0, 0)):
    exp_name = config.get("exp_name", "NA")
    reduce_dim = config.get("reduce_dim", "NA")
    reduce_dim_method = config.get("reduce_dim_method", "NA")
    n_dim = config.get("n_dim", "NA")
    idf = config.get("idf", "NA")

    header_text = (
        f'exp_name: "{exp_name}"    '
        f"reduce_dim: {str(reduce_dim).lower()}    "
        f'reduce_dim_method: "{reduce_dim_method}"    '
        f"n_dim: {n_dim}    "
        f"idf: {str(idf).lower()}"
    )

    # Bigger font (auto-scale with width)
    font_size = max(48, int(width * 0.035))
    try:
        font = ImageFont.truetype("arial.ttf", size=font_size)
    except Exception:
        font = ImageFont.load_default()

    draw_dummy = ImageDraw.Draw(Image.new("RGB", (width, 10), bg))
    max_chars = max(20, int(width / (font_size * 0.55)))
    lines = textwrap.wrap(header_text, width=max_chars) or [header_text]

    line_heights = []
    for line in lines:
        bbox = draw_dummy.textbbox((0, 0), line, font=font)
        line_heights.append(bbox[3] - bbox[1])

    text_h = sum(line_heights) + int((len(lines) - 1) * (pad_y * 0.4))
    header_h = pad_y * 2 + text_h

    header = Image.new("RGB", (width, header_h), bg)
    draw = ImageDraw.Draw(header)

    draw.line([(0, header_h - 2), (width, header_h - 2)], fill=(220, 220, 220), width=3)

    y = pad_y
    for line in lines:
        bbox = draw.textbbox((0, 0), line, font=font)
        h = bbox[3] - bbox[1]
        draw.text((pad_x, y), line, fill=fg, font=font)
        y += h + int(pad_y * 0.4)

    return header

def df_to_image(df, title="", dpi=300):
    df = df.astype(str)
    nrows, ncols = df.shape

    # --- Column width calculation (no changes here) ---
    max_lens = [
        max(df[col].str.len().max(), len(col))
        for col in df.columns
    ]
    total = sum(max_lens) if sum(max_lens) > 0 else 1
    col_widths = [l / total for l in max_lens]

    # --- REVISED: More precise figure size calculation ---
    # This is the key to reducing whitespace. We make the figure height
    # proportional to the number of rows.
    # The '0.65' is an empirical factor for row height + padding.
    # The '+ 1.0' at the end adds dedicated space for the title.
    # **You can tweak these values for fine control.**
    fig_w = max(16, total * 0.20)
    fig_h = (nrows + 1) * 0.65 + 1.0
    
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
    ax.axis("off")

    # --- REVISED: Use fig.suptitle for a more robust title ---
    if title:
        # The `y` coordinate controls vertical position (1.0 is top of figure)
        # Tweak this value (e.g., 0.97, 0.95) to adjust space between title and table
        fig.suptitle(
            title,
            fontsize=22, # Matched header font size
            fontweight="bold",
            y=0.98
        )

    # Place the table in the center of our now much smaller figure area
    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        loc="center", # Reverted to 'center'
        cellLoc="left",
        colWidths=col_widths
    )

    table.auto_set_font_size(False)
    table.set_fontsize(20)
    table.scale(1, 2.0)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_fontsize(22)
            cell.set_text_props(weight="bold")

    # Use a small pad_inches to ensure content isn't cut off by bbox_inches='tight'
    # This prevents the overlap issue.
    tmp = RUN_DIR / "__tmp_df.png"
    fig.savefig(tmp, bbox_inches="tight", dpi=dpi, pad_inches=0.1)
    plt.close(fig)

    im = Image.open(tmp).convert("RGB")
    tmp.unlink(missing_ok=True)

    return im

# -----------------------------
# Convert to images
# -----------------------------
# NEW: add all rows from this file as a df image ABOVE the other two
df0 = pd.read_excel("output/local_flavordb_similarity.xlsx")
df0 = df0.applymap(lambda x: f"{x:.3f}" if isinstance(x, (int, float)) else x)
df1_img_title = Path(xlsx_files[0]).name
df2_img_title = Path(xlsx_files[1]).name

df_img0 = df_to_image(df0, title="local_flavordb_similarity.xlsx", dpi=config["dpi"])
df_img1 = df_to_image(df1, title=df1_img_title, dpi=config["dpi"])
df_img2 = df_to_image(df2, title=df2_img_title, dpi=config["dpi"])

# -----------------------------
# Resize all to same width
# -----------------------------
def resize_to_width(im, w):
    h = int(im.size[1] * w / im.size[0])
    return im.resize((w, h), Image.LANCZOS)

panel_w = min(
    img1.size[0], img2.size[0],
    df_img0.size[0], df_img1.size[0], df_img2.size[0]
)

img1 = resize_to_width(img1, panel_w)
img2 = resize_to_width(img2, panel_w)
# df_img0 = resize_to_width(df_img0, panel_w * 2)
# df_img1 = resize_to_width(df_img1, panel_w * 2)
# df_img2 = resize_to_width(df_img2, panel_w * 2)

# -----------------------------
# Create composite
# -----------------------------
gap = 2
df_gap = gap 
pad = 60
bg = (255, 255, 255)

header = make_header_image(config, width=pad * 2 + panel_w * 2 + gap, dpi=config["dpi"])

row1_h = max(img1.size[1], img2.size[1])

canvas_w = pad * 2 + panel_w * 2 + gap
canvas_h = (
    header.size[1]
    + pad * 2
    + row1_h
    + gap
    + df_img0.size[1]
    + gap
    + df_img1.size[1]
    + gap
    + df_img2.size[1]
)

canvas = Image.new("RGB", (canvas_w, canvas_h), bg)

# -----------------------------
# Paste images
# -----------------------------
y0 = header.size[1]

canvas.paste(header, (0, 0))

# Top row (plots)
canvas.paste(img1, (pad, y0 + pad))
canvas.paste(img2, (pad + panel_w + gap, y0 + pad))

# Dataframes stacked
y_df0 = y0 + pad + row1_h + df_gap
canvas.paste(df_img0, (pad, y_df0))

y_df1 = y_df0 + df_img0.size[1] + df_gap
canvas.paste(df_img1, (pad, y_df1))

y_df2 = y_df1 + df_img1.size[1] + df_gap
canvas.paste(df_img2, (pad, y_df2))

# -----------------------------
# Save
# -----------------------------
out_path = RUN_DIR / f"{run_name}_summary.png"
out_path.parent.mkdir(exist_ok=True)

canvas.save(out_path, dpi=(config["dpi"], config["dpi"]))
canvas.save(Path("runs") / f"{run_name}_summary.png", dpi=(config["dpi"], config["dpi"]))

print(f"Saved composite image to: {out_path}")

C:\Users\jpcle8\AppData\Local\Temp\ipykernel_24608\2853101120.py:194: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df0 = df0.applymap(lambda x: f"{x:.3f}" if isinstance(x, (int, float)) else x)


Saved composite image to: runs\GPT52-RAG_rd-true_meth-svd_dim-50_idf-false_2026-03-05_14-37\GPT52-RAG_rd-true_meth-svd_dim-50_idf-false_2026-03-05_14-37_summary.png


In [197]:
# -----------------------------
# Store df0 results across runs
# -----------------------------
history_file = Path("runs") / "local_flavordb_similarity_history.xlsx"

# add run identifier
df0_history = df0.copy(deep=True)
df0_history.insert(0, "run_name", run_name)

if history_file.exists():
    prev = pd.read_excel(history_file)
    df_all = pd.concat([prev, df0_history], ignore_index=True)
else:
    df_all = df0_history

df_all.to_excel(history_file, index=False)

# Searching